# Caso d'uso: l'evento di maltempo del 13–17 novembre 2026 in Campania (WRF meteo@uniparthenope)

**Dati**: uscite orarie del modello WRF del Centro meteo@uniparthenope, dominio **d03**
(Campania, risoluzione ~3 km), serie *history*, scaricate dal server OPeNDAP del Centro con
`scripts/fetch_wrf_d03.py` nell'area *raw* della piattaforma (sola lettura).

**Flusso**: dato raw → Xarray + Dask (apertura di ~120 file orari) → temperatura a 2 m, vento a 10 m,
precipitazione oraria e cumulata → serie nelle città → prodotto *curated* Zarr → dashboard Voilà.

Il notebook è parametrizzato (cella `parameters`) ed eseguibile con Papermill.

In [ ]:
data_dir = "/home/jovyan/data/raw/models/wrf5_d03"
output_dir = "/home/jovyan/work/outputs/wrf_campania"
start = "2026-11-13T00"
end = "2026-11-17T23"
focus_city = "Positano"
time_chunk = 24
n_workers = 2

In [ ]:
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import dask
from dask.distributed import Client, LocalCluster

CITTA = {  # coordinate approssimate dei centri urbani
    "Napoli": (40.852, 14.268), "Salerno": (40.678, 14.765), "Avellino": (40.914, 14.793),
    "Benevento": (41.130, 14.782), "Caserta": (41.072, 14.333), "Positano": (40.628, 14.485),
    "Capri": (40.551, 14.243),
}
timings, t_total = {}, time.perf_counter()
out = Path(output_dir); (out / "figure").mkdir(parents=True, exist_ok=True)
print("xarray", xr.__version__, "| dask", dask.__version__)

## 1. Dask e apertura lazy dei file orari

In [ ]:
t0 = time.perf_counter()
cluster = LocalCluster(n_workers=n_workers, threads_per_worker=1, memory_limit="1GB",
                       dashboard_address=None)
client = Client(cluster)
timings["avvio_dask_s"] = time.perf_counter() - t0

t0 = time.perf_counter()
files = sorted(Path(data_dir).glob("wrf5_d03_*Z*.nc"))
grid = xr.open_dataset(Path(data_dir) / "wrf5_d03_grid.nc")
ds = xr.open_mfdataset(files, combine="nested", concat_dim="time", parallel=False,
                       chunks={}, data_vars="minimal", coords="minimal", compat="override")
ds = ds.sortby("time").sel(time=slice(start, end)).chunk({"time": time_chunk})
lat, lon = grid.XLAT.values, grid.XLONG.values
land = grid.LANDMASK.values > 0.5
timings["apertura_lazy_s"] = time.perf_counter() - t0

ore_attese = pd.date_range(start, end, freq="h")
ore_mancanti = sorted(set(ore_attese) - set(pd.to_datetime(ds.time.values)))
print(f"{len(files)} file, {ds.sizes['time']} ore nel periodo, griglia "
      f"{ds.sizes['south_north']}x{ds.sizes['west_east']}, ore mancanti: {len(ore_mancanti)}")

## 2. Grandezze derivate
* temperatura a 2 m in °C;
* velocità del vento a 10 m: il modulo di (U10, V10) non dipende dalla rotazione della griglia WRF;
* precipitazione oraria: le variabili `RAINC` + `RAINNC` sono **cumulate dall'inizio di ogni corsa**
  del modello; la serie *history* unisce corse successive, per cui la cumulata può ripartire da zero.
  La pioggia oraria è la differenza tra ore consecutive; dove la differenza è negativa (ripartenza)
  si usa il valore cumulato dell'ora stessa. Il numero di ripartenze è riportato nei risultati.

In [ ]:
t0 = time.perf_counter()
t2c = ds.T2 - 273.15
ws = np.hypot(ds.U10, ds.V10)
acc = ds.RAINC + ds.RAINNC
dacc = acc.diff("time")
rain_h = xr.where(dacc >= 0, dacc, acc.isel(time=slice(1, None)))
rain_h = rain_h.where(rain_h >= 0, 0)  # protezione da valori spuri
n_reset = int((dacc.mean(("south_north", "west_east")) < 0).sum().compute())
rain_tot = rain_h.sum("time")

landm = xr.DataArray(land, dims=("south_north", "west_east"))
t2_mean_map, ws_max_map, rain_tot_map, t2_land_ts, t2_sea_ts, ws_land_ts, rain_land_ts = dask.compute(
    t2c.mean("time"), ws.max("time"), rain_tot,
    t2c.where(landm).mean(("south_north", "west_east")),
    t2c.where(~landm).mean(("south_north", "west_east")),
    ws.where(landm).mean(("south_north", "west_east")),
    rain_h.where(landm).mean(("south_north", "west_east")))
timings["calcolo_campi_s"] = time.perf_counter() - t0

## 3. Serie temporali nelle città (punto di griglia più vicino)

In [ ]:
t0 = time.perf_counter()
def nearest(la, lo):
    d = (lat - la) ** 2 + ((lon - lo) * np.cos(np.deg2rad(la))) ** 2
    return np.unravel_index(np.argmin(d), d.shape)

idx = {c: nearest(*xy) for c, xy in CITTA.items()}
sel = {c: dict(south_north=j, west_east=i) for c, (j, i) in idx.items()}
serie = {c: dask.compute(t2c.isel(**s), ws.isel(**s), rain_h.isel(**s)) for c, s in sel.items()}
timings["serie_citta_s"] = time.perf_counter() - t0

def argmax_ll(a):
    j, i = np.unravel_index(np.nanargmax(a), a.shape)
    return round(float(lat[j, i]), 2), round(float(lon[j, i]), 2)

stats = {
    "dati": "WRF5 d03 meteo@uniparthenope (history)",
    "periodo": f"{pd.Timestamp(ds.time.values[0]):%Y-%m-%d %H}:00 - {pd.Timestamp(ds.time.values[-1]):%Y-%m-%d %H}:00",
    "ore_disponibili": int(ds.sizes["time"]), "ore_mancanti": len(ore_mancanti),
    "griglia": f"{ds.sizes['south_north']}x{ds.sizes['west_east']}",
    "ripartenze_cumulata": n_reset,
    "t2_media_terra_C": round(float(t2_land_ts.mean()), 1),
    "t2_media_mare_C": round(float(t2_sea_ts.mean()), 1),
    "vento_max_ms": round(float(np.nanmax(ws_max_map)), 1),
    "vento_max_posizione": argmax_ll(ws_max_map.values),
    "pioggia_cumulata_media_terra_mm": round(float(rain_tot_map.where(landm).mean()), 1),
    "pioggia_cumulata_max_mm": round(float(np.nanmax(rain_tot_map)), 1),
    "pioggia_cumulata_max_posizione": argmax_ll(rain_tot_map.values),
    "ora_pioggia_max_terra": f"{pd.Timestamp(rain_land_ts.idxmax().values):%Y-%m-%d %H}:00",
    "citta": {c: {"t2_media_C": round(float(s[0].mean()), 1), "t2_min_C": round(float(s[0].min()), 1),
                  "t2_max_C": round(float(s[0].max()), 1), "vento_max_ms": round(float(s[1].max()), 1),
                  "pioggia_tot_mm": round(float(s[2].sum()), 1),
                  "pioggia_max_oraria_mm": round(float(s[2].max()), 1)}
              for c, s in serie.items()},
}
print(json.dumps(stats, indent=2, ensure_ascii=False))

## 4. Visualizzazioni

In [ ]:
t0 = time.perf_counter()
plt.rcParams.update({"figure.dpi": 120, "font.size": 10})
def mappa(campo, titolo, etichetta, cmap, nome, **kw):
    fig, ax = plt.subplots(figsize=(7, 6))
    m = ax.pcolormesh(lon, lat, campo, cmap=cmap, shading="auto", **kw)
    ax.contour(lon, lat, land.astype(float), levels=[0.5], colors="k", linewidths=0.6)
    for c, (la, lo) in CITTA.items():
        ax.plot(lo, la, "k.", ms=4); ax.annotate(c, (lo, la), fontsize=7, xytext=(3, 3),
                                                 textcoords="offset points")
    fig.colorbar(m, ax=ax, label=etichetta, shrink=.85)
    ax.set(title=titolo, xlabel="Longitudine [°E]", ylabel="Latitudine [°N]"); ax.set_aspect("equal")
    fig.tight_layout(); fig.savefig(out / "figure" / nome, dpi=200); plt.show()

periodo = stats["periodo"]
mappa(rain_tot_map, f"Precipitazione cumulata\n{periodo}", "mm", "Blues", "fig1_pioggia_cumulata.png")
mappa(ws_max_map, f"Velocità massima del vento a 10 m\n{periodo}", "m/s", "YlOrRd", "fig2_vento_max.png")
mappa(t2_mean_map, f"Temperatura media a 2 m\n{periodo}", "°C", "RdYlBu_r", "fig3_t2_media.png")

fig, axs = plt.subplots(3, 1, figsize=(9, 8), sharex=True)
for c in ("Napoli", "Salerno", "Avellino", focus_city):
    s = serie[c]
    axs[0].plot(s[0].time, s[0], label=c); axs[1].plot(s[1].time, s[1], label=c)
rf = serie[focus_city][2]
axs[2].bar(rf.time.values, rf.values, width=1 / 24, color="tab:blue")
axs[0].set_ylabel("T 2 m [°C]"); axs[1].set_ylabel("Vento 10 m [m/s]")
axs[2].set_ylabel(f"Pioggia oraria\n{focus_city} [mm]"); axs[0].legend(ncol=4, fontsize=8)
for a in axs: a.grid(alpha=.3)
fig.suptitle("Serie orarie nelle città (punto di griglia più vicino)"); fig.tight_layout()
fig.savefig(out / "figure" / "fig4_serie_citta.png", dpi=200); plt.show()
timings["grafici_s"] = time.perf_counter() - t0

## 5. Prodotto curato in Zarr

In [ ]:
t0 = time.perf_counter()
curated = xr.Dataset(
    {"t2c": t2c.isel(time=slice(1, None)), "ws10": ws.isel(time=slice(1, None)), "rain_h": rain_h,
     "rain_tot": rain_tot_map, "t2_mean": t2_mean_map, "ws_max": ws_max_map,
     "lat": (("south_north", "west_east"), lat), "lon": (("south_north", "west_east"), lon),
     "landmask": (("south_north", "west_east"), land.astype("int8"))},
    attrs={"title": "Evento 13-17 novembre 2026, Campania - prodotto curato wxhub",
           "source": "WRF5 d03 meteo@uniparthenope (history), dati aperti IODL 2.0",
           "history": f"creato da caso_uso_wrf_campania.ipynb il {pd.Timestamp.now():%Y-%m-%d}"})
zarr_path = out / "wrf_campania_curated.zarr"
curated.chunk({"time": time_chunk}).to_zarr(zarr_path, mode="w")
chk = xr.open_zarr(zarr_path)
assert np.allclose(chk.rain_tot.values, rain_tot_map.values, equal_nan=True)
timings["scrittura_zarr_s"] = time.perf_counter() - t0
size_mb = sum(f.stat().st_size for f in zarr_path.rglob("*") if f.is_file()) / 1e6
print(f"Zarr: {zarr_path} ({size_mb:.1f} MB), verificato")

In [ ]:
timings["totale_s"] = time.perf_counter() - t_total
stats["zarr_MB"] = round(size_mb, 1)
stats["tempi_s"] = {k: round(v, 2) for k, v in timings.items()}
(out / "risultati.json").write_text(json.dumps(stats, indent=2, ensure_ascii=False))
print(json.dumps(stats["tempi_s"], indent=2))
client.close(); cluster.close()